# NISAR GCOV — Metadata and Auxiliary Layers


## Before exporting

Worth a quick geographic look at the native subset before writing GeoTIFF layers from it.

# 09 — Advanced GCOV: Frequencies, Masks & Batch Terms

Batch mask and statistics generation across terms, plus the first half of GeoTIFF export — Module 10 finishes the job with QGIS validation.

The GCOV `mask` dataset is read once on the persisted subset and applied to every exported real-valued term before it's written -- `MASK == 0` pixels are NaN in the exported GeoTIFF, not just flagged as nodata in the file's metadata while the underlying value stays whatever the raw dataset happened to contain.

In [ ]:
from pathlib import Path
from nisar_utils.bootstrap import setup_workshop
WORKSHOP_ROOT = setup_workshop()

from nisar_utils.config import load_config
import numpy as np
import rasterio
from affine import Affine
from nisar_utils.gcov import open_gcov, get_grid_coordinates, read_window

from nisar_utils.workflow import (
    build_profile, resolve_frequency, resolve_terms,
    resolve_aoi, resolve_window
)

cfg = load_config()
NISAR_FILE = Path(cfg["nisar_file"])
profile = build_profile(cfg)
freq = resolve_frequency(cfg, profile)
terms, diagonal_terms, off_diagonal_terms = resolve_terms(profile, freq)

print("File:", NISAR_FILE)
print("SAR family:", profile.sar_family)
print("Band:", profile.band)
print("Level:", profile.product_level)
print("Product:", profile.product_type)
print("GCOV root:", profile.gcov_root)
print("Frequency:", freq)
print("Polarization:", profile.polarization_channels)


In [ ]:
from IPython.display import display

try:
    from nisar_utils.gcov import open_gcov, get_grid_coordinates
    from nisar_utils.mapping import folium_scene_map

    _grid9 = f"{profile.gcov_root}/grids/{freq}"

    with open_gcov(NISAR_FILE) as _f9:
        _x9, _y9 = get_grid_coordinates(_f9, _grid9)

    print("Grid:", _grid9)
    print("X coordinates:", len(_x9))
    print("Y coordinates:", len(_y9))
    print("EPSG:", profile.epsg)

    # ------------------------------------------------------------
    # Normalize Module 06 geographic AOI
    # ------------------------------------------------------------
    _aoi9 = cfg.get("default_aoi") or cfg.get("aoi")

    print("Stored AOI:", _aoi9)

    if _aoi9 and all(k in _aoi9 for k in
                     ("xmin", "xmax", "ymin", "ymax")):

        _map_aoi9 = {
            "lon_min": float(_aoi9["xmin"]),
            "lon_max": float(_aoi9["xmax"]),
            "lat_min": float(_aoi9["ymin"]),
            "lat_max": float(_aoi9["ymax"]),
        }

    elif _aoi9 and all(k in _aoi9 for k in
                       ("lon_min", "lon_max", "lat_min", "lat_max")):

        _map_aoi9 = {
            "lon_min": float(_aoi9["lon_min"]),
            "lon_max": float(_aoi9["lon_max"]),
            "lat_min": float(_aoi9["lat_min"]),
            "lat_max": float(_aoi9["lat_max"]),
        }

    else:
        _map_aoi9 = None

    print("Mapping AOI:", _map_aoi9)

    # ------------------------------------------------------------
    # Create map
    # ------------------------------------------------------------
    _m9 = folium_scene_map(
        _x9,
        _y9,
        profile.epsg,
        title="NISAR native subset — GeoTIFF context",
        aoi=_map_aoi9
    )

    print("Map object created:", type(_m9))

    # ------------------------------------------------------------
    # Explicitly render the Folium map
    # ------------------------------------------------------------
    display(_m9)

except Exception as e:
    print("Map preview unavailable:")
    print(type(e).__name__, ":", e)


In [ ]:
from nisar_utils.gcov import open_gcov
from nisar_utils.hdf5 import inspect_dataset

with open_gcov(NISAR_FILE) as f:
    for this_freq in profile.frequencies:
        grid=f"{profile.gcov_root}/grids/{this_freq}"
        discovered=profile.covariance_terms.get(this_freq,[])
        print("\nFrequency:",this_freq)
        print("Covariance terms:",discovered)
        for aux in ["mask","numberOfLooks","rtcGammaToSigmaFactor","inputDataExceptionMask"]:
            path=f"{grid}/{aux}"
            if path in f:
                info=inspect_dataset(f,path)
                print("Available:",aux,info["shape"],info["dtype"])
            else:
                print("Not present:",aux)
print("\nModule 09 STATUS: PASS")


### Native-resolution GeoTIFF export

- Uses the exact r0/r1/c0/c1 subset from Module 06 — no new AOI.
- Only real-valued covariance terms get exported; complex ones are skipped.
- Reads the GCOV MASK on the same subset and writes MASK==0 pixels as NaN, not just as a declared nodata value in the file metadata.
- Keeps the product's native projected CRS/EPSG as-is, with no resampling or reprojection.
- Lands in the same acquisition folder Module 08 created.
- Ignores Module 08's visualization downsampling — this is full resolution.

In [ ]:

# ------------------------------------------------------------------
# Use the EXACT same acquisition-folder identity as Module 08 v10.
# For a filename such as:
# NISAR_L2_UR_GCOV_029_048_D_074_2005_QPDH_A_20260828T125812...
# Track=029, Frame=048, Pass=D, Date=20260828.
#
# If Module 08 has already created the folder, Module 09 reuses it.
# Otherwise Module 09 creates the same folder name.
# ------------------------------------------------------------------
import re
from pathlib import Path

_filename = NISAR_FILE.stem

_m = re.search(
    r"(?P<cycle>\d{3})_(?P<track>\d{3})_(?P<pass>[AD])_"
    r"(?P<frame>\d{3})_(?P<mode>[^_]+)_(?P<pole>[^_]+)_"
    r"(?P<source>[AM])_"
    r"(?P<start>\d{8}T\d{6})_"
    r"(?P<stop>\d{8}T\d{6})",
    _filename,
    flags=re.IGNORECASE,
)

if not _m:
    raise RuntimeError(
        "Could not parse Track, Frame, A/D pass and acquisition date "
        "from the NISAR filename using the Module 08 naming convention."
    )

cycle = _m.group("cycle")
track = _m.group("track")
pass_code = _m.group("pass").upper()
frame = _m.group("frame")
date_compact = _m.group("start")[:8]

sar_token = str(getattr(profile, "sar_family", "SAR") or "SAR")
sar_token = re.sub(r"[^A-Za-z0-9.-]+", "_", sar_token).strip("_")

UNIQUE_OUTPUT_NAME = (
    f"NISAR_{sar_token}_{track}_{frame}_{pass_code}_{date_compact}"
)

# If Module 08's folder is already present, this points to that exact folder.
# Otherwise create the same location/name used by Module 08.
UNIQUE_OUTPUT_DIR = (
    WORKSHOP_ROOT / "outputs" / "module_08_visualization" / UNIQUE_OUTPUT_NAME
)
UNIQUE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Shared Module 08/09 acquisition folder:", UNIQUE_OUTPUT_DIR)
print("Acquisition ID:", UNIQUE_OUTPUT_NAME)
print("Cycle:", cycle)
print("Track:", track)
print("Frame:", frame)
print("Pass:", pass_code)
print("Pass date:", date_compact)


In [ ]:

# Module 06 is authoritative for the native NISAR AOI subset.
# Do not recalculate it from the persisted WGS84 AOI.
spatial_cfg = cfg.get("spatial_subset")

if not isinstance(spatial_cfg, dict) or not all(
    k in spatial_cfg for k in ("r0", "r1", "c0", "c1")
):
    raise RuntimeError(
        "No complete persisted native AOI subset was found. "
        "Run Module 06 successfully before Module 09."
    )

r0, r1, c0, c1 = map(
    int,
    (
        spatial_cfg["r0"],
        spatial_cfg["r1"],
        spatial_cfg["c0"],
        spatial_cfg["c1"],
    ),
)

grid = f"{profile.gcov_root}/grids/{freq}"

with open_gcov(NISAR_FILE) as f:
    x, y = get_grid_coordinates(f, grid)

    if not (0 <= r0 < r1 <= len(y) and 0 <= c0 < c1 <= len(x)):
        raise ValueError(
            f"Persisted AOI {(r0, r1, c0, c1)} is outside the GCOV grid "
            f"{(len(y), len(x))}."
        )

    dx = (
        float(abs(x[1] - x[0]))
        if len(x) > 1
        else float(getattr(profile, "x_spacing", 1.0) or 1.0)
    )
    dy = (
        float(abs(y[1] - y[0]))
        if len(y) > 1
        else float(getattr(profile, "y_spacing", 1.0) or 1.0)
    )

    if not np.isfinite(dx) or dx <= 0 or not np.isfinite(dy) or dy <= 0:
        raise ValueError(f"Invalid native grid spacing: dx={dx}, dy={dy}")

    # Keep the established training-package convention used by the existing
    # GeoTIFF exporter: x/y grid coordinates define the upper-left map
    # coordinate of the selected PixelIsArea grid.
    left = float(x[c0])
    top = float(y[r0])
    transform = Affine(dx, 0.0, left, 0.0, -dy, top)

epsg = getattr(profile, "epsg", None)
if epsg is None:
    raise ValueError(
        "Native NISAR EPSG is unavailable; GeoTIFF export cannot proceed safely."
    )

export_crs = f"EPSG:{int(epsg)}"
full_shape = (r1 - r0, c1 - c0)

print("Frequency:", freq)
print("Native CRS:", export_crs)
print("Native transform:", transform)
print("Native AOI window:", (r0, r1, c0, c1))
print("Native AOI shape:", full_shape)
print("Output folder:", UNIQUE_OUTPUT_DIR)


In [ ]:

# Export every discovered real-valued GCOV covariance term.
# Complex-valued channels are skipped for ALL product/polarization modes.
from nisar_utils.gcov import read_gcov_mask, apply_gcov_mask

with open_gcov(NISAR_FILE) as f:
    invalid_mask, mask_arr = read_gcov_mask(f, grid, r0, r1, c0, c1)

print("GCOV MASK: invalid pixels in AOI:", f"{int(np.count_nonzero(invalid_mask)):,}",
      "of", f"{invalid_mask.size:,}")

def _safe_tiff_dtype(dtype):
    dt = np.dtype(dtype)
    if dt.kind == "c":
        return None
    if dt.kind in {"b", "i", "u", "f"}:
        return dt
    return None


def _nodata_from_dataset(ds, dtype):
    fill = ds.attrs.get("_FillValue", None)

    if isinstance(fill, np.ndarray) and fill.size == 1:
        fill = fill.reshape(-1)[0]

    if isinstance(fill, bytes):
        try:
            fill = fill.decode()
        except Exception:
            pass

    if np.issubdtype(dtype, np.floating):
        if fill is None:
            return np.nan
        try:
            return float(fill)
        except Exception:
            return np.nan

    if fill is not None:
        try:
            return np.asarray(fill, dtype=dtype).item()
        except Exception:
            return None

    return None


def write_native_geotiff(term, output_path):
    dataset_path = f"{grid}/{term}"

    with open_gcov(NISAR_FILE) as f:
        if dataset_path not in f:
            raise KeyError(f"GCOV dataset not found: {dataset_path}")

        ds = f[dataset_path]
        source_dtype = np.dtype(ds.dtype)

        if np.issubdtype(source_dtype, np.complexfloating):
            return None, "complex"

        if not np.issubdtype(source_dtype, np.number):
            return None, f"unsupported dtype {source_dtype}"

        data = np.asarray(
            read_window(f, dataset_path, r0, r1, c0, c1)
        )

        if data.shape != full_shape:
            raise ValueError(
                f"Unexpected subset shape for {term}: "
                f"{data.shape}; expected {full_shape}."
            )

        dtype = _safe_tiff_dtype(data.dtype)
        if dtype is None:
            return None, f"unsupported dtype {data.dtype}"

        # Apply the GCOV validity mask before writing. A transmit-gap or
        # otherwise invalid pixel must not be exported as if it were real
        # data -- only *declaring* NaN as the nodata sentinel in the
        # GeoTIFF's metadata, without actually writing NaN into those
        # pixels, is not the same thing.
        if np.issubdtype(dtype, np.floating):
            data = apply_gcov_mask(data, term, invalid_mask)
            nodata = np.nan
        else:
            # Integer covariance terms can't represent NaN. This is a rare
            # case for continuous backscatter power data, but rather than
            # silently export unmasked integer pixels, fall back to the
            # dataset's own _FillValue if it has one.
            nodata = _nodata_from_dataset(ds, dtype)
            if nodata is not None:
                data = np.where(invalid_mask, nodata, data).astype(dtype)

    profile_kwargs = {
        "driver": "GTiff",
        "height": data.shape[0],
        "width": data.shape[1],
        "count": 1,
        "dtype": dtype.name,
        "crs": export_crs,
        "transform": transform,
        "compress": "deflate",
        "tiled": True,
    }

    if nodata is not None:
        profile_kwargs["nodata"] = nodata

    output_path = Path(output_path)

    with rasterio.open(output_path, "w", **profile_kwargs) as dst:
        dst.write(data, 1)
        dst.set_band_description(1, term)
        dst.update_tags(
            NISAR_PRODUCT="GCOV",
            NISAR_FREQUENCY=str(freq),
            NISAR_TERM=str(term),
            NISAR_NATIVE_EPSG=str(int(epsg)),
            NISAR_AOI_WINDOW=f"{r0},{r1},{c0},{c1}",
        )

    return output_path, "exported"


exported_paths = {}
skipped_complex = []
skipped_other = []

available_terms = list(
    profile.covariance_terms.get(freq, []) or []
)

print("\nNative GeoTIFF export terms:", available_terms)

for term in available_terms:
    output_path = UNIQUE_OUTPUT_DIR / f"{term}.tif"
    path, status = write_native_geotiff(term, output_path)

    if status == "exported":
        exported_paths[term] = path
        print(f"  EXPORTED: {term:8s} -> {path.name}")
    elif status == "complex":
        skipped_complex.append(term)
        print(
            f"  SKIPPED : {term:8s} -> complex-valued channel"
        )
    else:
        skipped_other.append((term, status))
        print(f"  SKIPPED : {term:8s} -> {status}")

print("\nReal-valued GeoTIFFs:", len(exported_paths))
print(
    "Complex channels skipped:",
    skipped_complex if skipped_complex else "none"
)
if skipped_other:
    print("Other skipped terms:", skipped_other)


In [ ]:

# QGIS-oriented validation and shared-folder check.
print("=" * 70)
print("MODULE 09 — NATIVE GEOTIFF EXPORT VALIDATION")
print("=" * 70)
print("Shared acquisition folder:", UNIQUE_OUTPUT_DIR)
print("Frequency:", freq)
print("Native CRS:", export_crs)
print("Expected native shape:", full_shape)
print("Expected AOI window:", (r0, r1, c0, c1))

for term, path in exported_paths.items():
    with rasterio.open(path) as src:
        print(f"\n{term}.tif")
        print("  shape    :", (src.height, src.width))
        print("  CRS      :", src.crs)
        print("  transform:", src.transform)
        print("  dtype    :", src.dtypes[0])
        print("  nodata   :", src.nodata)

        if (src.height, src.width) != full_shape:
            raise RuntimeError(
                f"{term}.tif shape does not match native AOI: {src.shape}"
            )

        if str(src.crs) != export_crs:
            raise RuntimeError(
                f"{term}.tif CRS does not match native NISAR CRS: {src.crs}"
            )

png_files = sorted(UNIQUE_OUTPUT_DIR.glob("*.png"))
tif_files = sorted(UNIQUE_OUTPUT_DIR.glob("*.tif"))

print("\nFiles currently in shared acquisition folder:")
for p in sorted(UNIQUE_OUTPUT_DIR.iterdir()):
    print(" ", p.name)

print("\nModule 08 PNG count :", len(png_files))
print("Module 09 TIFF count:", len(tif_files))

if png_files:
    print("Module 08 + Module 09 outputs coexist: PASS")
else:
    print(
        "NOTE: No PNG files are currently present in this acquisition folder. "
        "Run Module 08 v10 for this acquisition before Module 09 if the "
        "visualization PNGs are expected here."
    )

print("Native-resolution export: PASS")
print("Native NISAR PCS retained: PASS")
print("Complex-channel exclusion: PASS")
print("Shared acquisition folder: PASS")
print("QGIS-ready GeoTIFF export: PASS")
print("Module 09 STATUS: PASS")
